# Extract prospective ratios from [EnergyScope TD v2.2](https://github.com/energyscope/EnergyScope/tree/EnergyScope.py)

In [37]:
import pandas as pd

In [38]:
estd_data_path = 'Data/ESTD/'
results_path = 'Outputs/'

In [39]:
starting_year = 2020
years_list = [2025, 2030, 2035, 2040, 2045, 2050]

In [40]:
resources = pd.read_csv(estd_data_path+f'{starting_year}/Resources.csv', sep=';', header=2, usecols=[2,3,5], names=['ES_name', 'avail', 'c_op'])
tech = pd.read_csv(estd_data_path+f'{starting_year}/Technologies.csv', sep=';', usecols=[3,4,5], names=['ES_name', 'c_inv', 'c_maint'])
lyrio = pd.read_csv(estd_data_path+f'{starting_year}/Layers_in_out.csv', sep=';')

## Resources (availability and operation cost)

In [41]:
for year in years_list:
    new_df = pd.read_csv(estd_data_path+f'{year}/Resources.csv', sep=';', header=2, usecols=[2,3,5], names=['ES_name', 'avail', 'c_op'])
    resources = pd.merge(resources, new_df, on='ES_name', suffixes=('', f'_{year}'))
    resources[f'Ratio avail {year}/{starting_year}'] = resources[f'avail_{year}'] / resources['avail']
    resources[f'Ratio c_op {year}/{starting_year}'] = resources[f'c_op_{year}'] / resources['c_op']

In [42]:
c_op = resources[['ES_name', 'c_op'] + [f'c_op_{year}' for year in years_list] + [f'Ratio c_op {year}/{starting_year}' for year in years_list]].dropna()

In [43]:
c_op.rename(columns={'c_op': f'c_op_{starting_year}'}, inplace=True)
c_op['param'] = len(c_op) * ['c_op']
c_op.rename(columns={f'c_op_{year}': f'YEAR_{year}' for year in [starting_year] + years_list}, inplace=True)
c_op.rename(columns={f'Ratio c_op {year}/{starting_year}': f'Ratio {year}/{starting_year}' for year in years_list}, inplace=True)

## Technologies (investment and maintenance cost)

In [44]:
tech.drop(index=[0,1], inplace=True)
for year in years_list:
    new_df = pd.read_csv(estd_data_path+f'{year}/Technologies.csv', sep=';', usecols=[3,4,5], names=['ES_name', 'c_inv', 'c_maint'])
    new_df.drop(index=[0,1], inplace=True)
    tech = pd.merge(tech, new_df, on='ES_name', suffixes=('', f'_{year}'))

tech[['c_inv', 'c_maint'] + [f'c_inv_{year}' for year in years_list] + [f'c_maint_{year}' for year in years_list]] = tech[['c_inv', 'c_maint'] + [f'c_inv_{year}' for year in years_list] + [f'c_maint_{year}' for year in years_list]].astype(float)

for year in years_list:
    tech[f'Ratio c_inv {year}/{starting_year}'] = tech[f'c_inv_{year}'] / tech['c_inv']
    tech[f'Ratio c_maint {year}/{starting_year}'] = tech[f'c_maint_{year}'] / tech['c_maint']

In [45]:
c_inv = tech[['ES_name', 'c_inv'] + [f'c_inv_{year}' for year in years_list] + [f'Ratio c_inv {year}/{starting_year}' for year in years_list]].dropna()
c_inv['param'] = len(c_inv) * ['c_inv']
c_inv.rename(columns={'c_inv': f'c_inv_{starting_year}'}, inplace=True)
c_inv.rename(columns={f'c_inv_{year}': f'YEAR_{year}' for year in [starting_year] + years_list}, inplace=True)
c_inv.rename(columns={f'Ratio c_inv {year}/{starting_year}': f'Ratio {year}/{starting_year}' for year in years_list}, inplace=True)

In [46]:
c_maint = tech[['ES_name', 'c_maint'] + [f'c_maint_{year}' for year in years_list] + [f'Ratio c_maint {year}/{starting_year}' for year in years_list]].dropna()
c_maint['param'] = len(c_maint) * ['c_maint']
c_maint.rename(columns={'c_maint': f'c_maint_{starting_year}'}, inplace=True)
c_maint.rename(columns={f'c_maint_{year}': f'YEAR_{year}' for year in [starting_year] + years_list}, inplace=True)
c_maint.rename(columns={f'Ratio c_maint {year}/{starting_year}': f'Ratio {year}/{starting_year}' for year in years_list}, inplace=True)

## Efficiencies (layers_in_out)

In [47]:
lyrio = lyrio.melt(id_vars='param layers_in_out:').rename(columns={'variable': 'Flow', 'value': 'Value'})
lyrio.drop(lyrio[lyrio['Value'] == 0].index, inplace=True)
for year in years_list:
    new_df = pd.read_csv(estd_data_path+f'{year}/Layers_in_out.csv', sep=';')
    new_df = new_df.melt(id_vars='param layers_in_out:').rename(columns={'variable': 'Flow', 'value': 'Value'})
    new_df.drop(new_df[new_df['Value'] == 0].index, inplace=True)
    lyrio = pd.merge(lyrio, new_df, on=['param layers_in_out:', 'Flow'], suffixes=('', f'_{year}'))
    lyrio[f'Ratio {year}/{starting_year}'] = lyrio[f'Value_{year}'] / lyrio['Value']

In [48]:
lyrio.rename(columns={'Value': f'Value_{starting_year}'}, inplace=True)
layers = lyrio[(lyrio[f'Value_{starting_year}'] != 1.0) & (lyrio[f'Value_{starting_year}'] != -1.0)][['param layers_in_out:', 'Flow'] + [f'Value_{year}' for year in [starting_year] + years_list] + [f'Ratio {year}/{starting_year}' for year in years_list]].dropna()
layers.rename(columns={'param layers_in_out:': 'ES_name'}, inplace=True)
layers['param'] = len(layers) * ['layers_in_out']
layers.rename(columns={f'Value_{year}': f'YEAR_{year}' for year in [starting_year] + years_list}, inplace=True)

## Gathering all data in a single dataframe

In [49]:
prospective_ratios = pd.concat([c_op, c_inv, c_maint])
prospective_ratios.ES_name = prospective_ratios.ES_name.str.lstrip()
prospective_ratios.ES_name = prospective_ratios.ES_name.str.rstrip()

In [50]:
prospective_ratios.to_csv(results_path+'prospective_ratios_parameters_from_ESTD.csv', index=False)

In [51]:
layers.to_csv(results_path+'prospective_ratios_lyrios_from_ESTD.csv', index=False)

## Retrieve data for ES-QC

In [52]:
from energyscope.models import Model
from energyscope.energyscope import Energyscope
from energyscope.result import postprocessing

In [53]:
path_model = '../ES_Snapshot/'

In [54]:
solver_options = {
    'solver': 'gurobi',
    'solver_msg': 0,
}

In [55]:
model = Model([
    ('mod', path_model+'QC_es_main.mod'),
    ('mod', path_model+'QC_objective_function.mod'),
    ('dat', path_model+'QC_data.dat'),
    ('dat', path_model+'QC_mob_techs_dist_B2D.dat'),
    ('dat', path_model+'QC_techs_B2D.dat'),
    ('dat', path_model+'QC_mob_params.dat'),
    ('dat', path_model+'QC_validation.dat'),
])

In [56]:
es = Energyscope(model=model, solver_options=solver_options)
results = postprocessing(es.calc())

Gurobi 12.0.3:

In [57]:
mobility_freight = results.sets['MODELS_OF_TECHNOLOGIES_OF_FREIGHTMOB_ALL_DISTANCES']
mobility_private = results.sets['MODELS_OF_TECHNOLOGIES_OF_PRIVATEMOB_ALL_DISTANCES']
mobility_public = results.sets['MODELS_OF_TECHNOLOGIES_OF_PUBLICMOB_ALL_DISTANCES']

In [58]:
mapping_tech_es = pd.read_csv(estd_data_path+'mapping_techs_estd_esqc.csv')

In [59]:
layers = layers.merge(mapping_tech_es, left_on='ES_name', right_on='ES_name_ESTD')

In [60]:
prospective_ratios = prospective_ratios.merge(mapping_tech_es, left_on='ES_name', right_on='ES_name_ESTD')

In [61]:
# Split the ES_name_ES_QC column by commas and create new rows
layers_expanded = layers.set_index(layers.columns.drop('ES_name_ES_QC', 1).tolist()).ES_name_ES_QC.str.split(', ', expand=True).stack().reset_index().rename(columns={0: 'ES_name_ES_QC'}).loc[:, layers.columns]

prospective_ratios_expanded = prospective_ratios.set_index(prospective_ratios.columns.drop('ES_name_ES_QC', 1).tolist()).ES_name_ES_QC.str.split(', ', expand=True).stack().reset_index().rename(columns={0: 'ES_name_ES_QC'}).loc[:, prospective_ratios.columns]

In [62]:
def add_mobility_sub_components(df, mobility_dict):
    to_remove = []
    for i in range(len(df)):
        tech_name = df['ES_name_ES_QC'].iloc[i]
        if tech_name in mobility_dict.keys():
            for mob_mode in mobility_dict[tech_name]:
                new_row = df.iloc[i].copy()
                new_row['ES_name_ES_QC'] = mob_mode
                df.loc[df.index.max()+1] = new_row
                to_remove.append(i)
    return df, to_remove

In [63]:
index_to_remove = []
layers_expanded, to_remove = add_mobility_sub_components(layers_expanded, mobility_freight)
index_to_remove.extend(to_remove)
layers_expanded, to_remove = add_mobility_sub_components(layers_expanded, mobility_private)
index_to_remove.extend(to_remove)
layers_expanded, to_remove = add_mobility_sub_components(layers_expanded, mobility_public)
index_to_remove.extend(to_remove)

layers_expanded.drop(index=index_to_remove, inplace=True)
layers_expanded.reset_index(drop=True, inplace=True)

In [64]:
# Add location and source
prospective_ratios_expanded['Location'] = 'EU'
prospective_ratios_expanded['Source'] = 'G. Limpens, “Generating energy transition pathways: application to Belgium,” PhD thesis, Université catholique de Louvain, 2021. Available: https://dial.uclouvain.be/pr/boreal/object/boreal:249196'
layers_expanded['Location'] = 'EU'
layers_expanded['Source'] = 'G. Limpens, “Generating energy transition pathways: application to Belgium,” PhD thesis, Université catholique de Louvain, 2021. Available: https://dial.uclouvain.be/pr/boreal/object/boreal:249196'

In [65]:
prospective_ratios_expanded[['ES_name_ES_QC', 'param'] + [f'Ratio {year}/{starting_year}' for year in years_list] + [f'YEAR_{year}' for year in [starting_year] + years_list] + ['Location', 'Source']].to_csv(results_path+'prospective_ratios_parameters_from_ESTD_for_ESQC.csv', index=False)

In [66]:
layers_expanded[['ES_name_ES_QC', 'Flow'] + [f'Ratio {year}/{starting_year}' for year in years_list] + [f'YEAR_{year}' for year in [starting_year] + years_list] + ['Location', 'Source']].to_csv(results_path+'prospective_ratios_lyrios_from_ESTD_for_ESQC.csv', index=False)

# Linear interpolation for other prospective parameters (CA/QC)

In [67]:
data = pd.read_csv('Data/ESQC/prospective_parameters_ca_qc_2050.csv')

In [68]:
target_year = 2050

In [69]:
data[f'YEAR_{starting_year}'] = data[f'YEAR_{starting_year}'].astype(float)
data[f'YEAR_{target_year}'] = data[f'YEAR_{target_year}'].astype(float)

In [70]:
for year in years_list:
    data[f'YEAR_{year}'] = data[f'YEAR_{starting_year}'] + ((year - starting_year) / (target_year - starting_year)) * (data[f'YEAR_{target_year}'] - data[f'YEAR_{starting_year}'])

In [71]:
for year in years_list:
    data[f'Ratio {year}/{starting_year}'] = data[f'YEAR_{year}'] / data[f'YEAR_{starting_year}']

In [72]:
data.to_csv(results_path+'prospective_ratios_parameters_from_other_sources_for_ESQC.csv', index=False)